# RFModelEva — Evaluation and Discussion (Random Forest)

## Setup

In [1]:
import joblib
import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from evaluation_utils import (
    load_yelp_split,
    print_evaluation,
    print_top_terms,
    validate_pipeline_labels,
)

PART_B_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "data" / "yelp_review_full_raw_30k.csv").exists())
DATA_FILE = PART_B_DIR / "data" / "yelp_clean.csv"
MODEL_FILE = PART_B_DIR / "models" / "rf_tuned.joblib"        # tuned model from Q3
BASELINE_FILE = PART_B_DIR / "models" / "rf_pipeline.joblib"  # untuned model from Q2

## Load tuned model

In [2]:
# --- Reload the tuned Random Forest and check it carries the expected 3-class labels ---
pipeline = joblib.load(MODEL_FILE)
validate_pipeline_labels(pipeline, "Random Forest")
print("Classes:", list(pipeline.named_steps["clf"].classes_))

Classes: ['negative', 'neutral', 'positive']


## Tuned model evaluation

In [3]:
# --- Recreate the same held-out test set used throughout Part B ---
df, X_train, X_test, y_train, y_test = load_yelp_split(DATA_FILE)

print_top_terms(pipeline, X_train, y_train, "Random Forest")
print_evaluation(pipeline, X_test, y_test, "Random Forest")

=== Top Terms Per Sentiment Class (Random Forest) ===



negative:
horrible, rude, bad, terrible, tell, no, manager, customer, ask, minute

neutral:
decent, pretty, okay, ok, average, good, bit, overall, though, nothing



positive:
great, love, delicious, amaze, awesome, favorite, excellent, friendly, best, perfect



=== Q4: Tuned Yelp Random Forest 3-Class Classification Report ===
              precision    recall  f1-score   support

    negative     0.7669    0.7621    0.7645      2400
     neutral     0.4464    0.4133    0.4293      1200
    positive     0.7584    0.7913    0.7745      2400

    accuracy                         0.7040      6000
   macro avg     0.6572    0.6556    0.6561      6000
weighted avg     0.6994    0.7040    0.7014      6000

=== Q4: Tuned Yelp Random Forest Summary ===
Accuracy:        0.7040
Macro Precision: 0.6572
Macro Recall:    0.6556
Macro F1-score:  0.6561
Weighted F1:     0.7014

=== Q4: Tuned Yelp Random Forest Confusion Matrix ===
[[1829  323  248]
 [ 347  496  357]
 [ 209  292 1899]]
Class order: ['negative', 'neutral', 'positive']


## Untuned vs tuned

In [4]:
# --- Side-by-side metrics for the hyperparameter discussion: what tuning bought us ---
def score_row(model, name):
    preds = model.predict(X_test)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_test, preds, average="macro", zero_division=0
    )
    weighted_f1 = precision_recall_fscore_support(
        y_test, preds, average="weighted", zero_division=0
    )[2]
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "Macro P": macro_p,
        "Macro R": macro_r,
        "Macro F1": macro_f1,
        "Weighted F1": weighted_f1,
    }


comparison = pd.DataFrame([
    score_row(joblib.load(BASELINE_FILE), "Untuned baseline (Q2)"),
    score_row(pipeline, "Tuned (Q3)"),
]).set_index("Model").round(4)

print("=== Q4: Random Forest - Untuned vs Tuned ===")
print(comparison.to_string())

gain = comparison.loc["Tuned (Q3)", "Macro F1"] - comparison.loc["Untuned baseline (Q2)", "Macro F1"]
print(f"\nMacro F1 gain from hyperparameter tuning: {gain:+.4f}")

=== Q4: Random Forest - Untuned vs Tuned ===
                       Accuracy  Macro P  Macro R  Macro F1  Weighted F1
Model                                                                   
Untuned baseline (Q2)     0.705   0.6725   0.5928    0.5407       0.6368
Tuned (Q3)                0.704   0.6572   0.6556    0.6561       0.7014

Macro F1 gain from hyperparameter tuning: +0.1154
